In [ ]:
import joblib
import numpy as np
import pandas as pd
import shap
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# ============= CONFIG =============
model_paths = {
    'mets': '../../output/ml/mets/best_rf_model_results.pkl',
    'crp': '../../output/ml/crp/best_rf_model_results.pkl',
    'ir': '../../output/ml/ir/best_rf_model_results.pkl',
}

# Saved data slpits from models
data_paths = {
    'mets': {'X_all': '../../output/ml/mets/X_all.pkl', 'X_test': '../../output/ml/mets/X_test.pkl', 'y_test': '../../output/ml/mets/y_test.pkl'},
    'crp':  {'X_all': '../../output/ml/crp/X_all.pkl',  'X_test': '../../output/ml/crp/X_test.pkl',  'y_test': '../../output/ml/crp/y_test.pkl'},
    'ir':   {'X_all': '../../output/ml/ir/X_all.pkl',   'X_test': '../../output/ml/ir/X_test.pkl',   'y_test': '../../output/ml/ir/y_test.pkl'},
}

# Full target files
y_source_paths = {
    'mets': {'path': '../../data/01/mets/mets_target.tsv', 'col': 'mets'},
    'crp':  {'path': '../../data/01/crp/crp_target.tsv', 'col': 'crp_class'},
    'ir':   {'path': '../../data/01/ir/ir_target.tsv', 'col': 'ir'},
}

TOP_K = 15
N_SPLITS = 5
RANDOM_STATE = 42

all_results = {}

# ============= LOOP OVER EACH OUTCOME =============
for outcome, model_path in model_paths.items():
    print(f"\n{'='*60}")
    print(f"Processing outcome: {outcome}")
    print(f"{'='*60}")

    # ---- Load saved model results ----
    model_results = joblib.load(model_path)
    best_params = model_results['best_params']
    clean_params = {k.replace('classifier__', ''): v for k, v in best_params.items()}

    # ---- Load saved artifacts ----
    X_all = joblib.load(data_paths[outcome]['X_all'])
    X_test_final = joblib.load(data_paths[outcome]['X_test'])
    y_test = joblib.load(data_paths[outcome]['y_test'])

    # ---- Reconstruct X_train_final by dropping test indices from X_all ----
    X_train_final = X_all.drop(index=X_test_final.index)
    print(f"Reconstructed X_train_final shape: {X_train_final.shape}")
    print(f"X_test_final shape: {X_test_final.shape}")
    print(f"X_all shape: {X_all.shape} (should equal train + test)")

    # Sanity check: no overlap, and train+test recovers all of X_all
    assert len(set(X_train_final.index) & set(X_test_final.index)) == 0, \
        "Overlap found between reconstructed train and test indices!"
    assert X_train_final.shape[0] + X_test_final.shape[0] == X_all.shape[0], \
        "Train + test row counts don't add up to X_all!"

    # ---- Load original full target file and align to recovered training indices ----
    y_source = y_source_paths[outcome]
    y_full = pd.read_csv(y_source['path'], sep='\t', index_col='SEQN')[y_source['col']]
    y_train = y_full.loc[X_train_final.index]

    # Sanity check: y_test should match what you already saved
    y_test_check = y_full.loc[X_test_final.index]
    assert (y_test_check.values == y_test.values).all(), \
        f"Reconstructed y_test doesn't match saved y_test for {outcome}! Check target file/column."

    print(f"Reconstructed y_train shape: {y_train.shape}, class distribution:\n{y_train.value_counts()}")

    # ---- Run 5-fold CV, computing SHAP values per fold ----
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_importances = []

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train_final, y_train)):
        X_fold_train = X_train_final.iloc[train_idx]
        y_fold_train = y_train.iloc[train_idx]

        fold_model = RandomForestClassifier(
            **clean_params, random_state=RANDOM_STATE, n_jobs=-1
        )
        fold_model.fit(X_fold_train, y_fold_train)

        explainer = shap.TreeExplainer(fold_model)
        shap_values = explainer.shap_values(X_fold_train)

        # Handle different SHAP output formats across versions:
        # - list of arrays (older versions): [class_0_array, class_1_array]
        # - 3D array (newer versions): (n_samples, n_features, n_classes)
        # - 2D array (some versions/binary setups): (n_samples, n_features)
        if isinstance(shap_values, list):
            mean_abs_shap = np.abs(shap_values[1]).mean(axis=0)
        elif shap_values.ndim == 3:
            mean_abs_shap = np.abs(shap_values[:, :, 1]).mean(axis=0)
        else:
            mean_abs_shap = np.abs(shap_values).mean(axis=0)

        fold_importances.append(mean_abs_shap)
        print(f"  Fold {fold_idx + 1}/{N_SPLITS} complete")
        
    # ---- Build importance dataframe ----
    importance_df = pd.DataFrame(fold_importances, columns=X_train_final.columns)

    # ---- Top-K features by mean importance ----
    mean_importance = importance_df.mean(axis=0).sort_values(ascending=False)
    top_features = mean_importance.head(TOP_K).index.tolist()

    print(f"\nTop {TOP_K} features (by mean SHAP importance across folds):")
    print(mean_importance.head(TOP_K))

    # ---- Rank stability restricted to top-K features ----
    top_importance_df = importance_df[top_features]
    rank_df = top_importance_df.rank(axis=1, ascending=False)
    rank_stability = rank_df.std(axis=0).sort_values()

    print(f"\nRank stability among top {TOP_K} features (std of rank, lower = more stable):")
    print(rank_stability)

    # ---- Pairwise Spearman correlation (top-K only) ----
    print(f"\nPairwise Spearman correlation between folds (top {TOP_K} features only):")
    spearman_results = []
    for i in range(N_SPLITS):
        for j in range(i + 1, N_SPLITS):
            corr, _ = spearmanr(top_importance_df.iloc[i], top_importance_df.iloc[j])
            spearman_results.append(corr)
            print(f"  Fold {i} vs Fold {j}: rho = {corr:.3f}")
    print(f"  Mean pairwise Spearman rho: {np.mean(spearman_results):.3f}")

    # ---- Top-10 overlap across all folds ----
    top10_per_fold = [set(importance_df.iloc[i].nlargest(10).index) for i in range(N_SPLITS)]
    common_top10 = set.intersection(*top10_per_fold)
    print(f"\nFeatures in top-10 across ALL folds: {len(common_top10)}")
    print(common_top10)

    all_results[outcome] = {
        'importance_df': importance_df,
        'top_features': top_features,
        'mean_importance': mean_importance,
        'rank_stability': rank_stability,
        'mean_spearman': np.mean(spearman_results),
        'common_top10': common_top10,
    }

# ============= CROSS-OUTCOME SUMMARY =============
print(f"\n{'='*60}")
print("SUMMARY ACROSS ALL OUTCOMES")
print(f"{'='*60}")
summary_df = pd.DataFrame({
    outcome: {
        'mean_spearman_rho': res['mean_spearman'],
        'n_stable_top10': len(res['common_top10']),
    }
    for outcome, res in all_results.items()
}).T
print(summary_df)

joblib.dump(all_results, 'shap_stability_all_outcomes.pkl')
print("\nSaved full results to shap_stability_all_outcomes.pkl")


Processing outcome: mets
Reconstructed X_train_final shape: (6645, 36)
X_test_final shape: (1662, 36)
X_all shape: (8307, 36) (should equal train + test)
Reconstructed y_train shape: (6645,), class distribution:
mets
0    4568
1    2077
Name: count, dtype: int64
  Fold 1/5 complete
  Fold 2/5 complete
  Fold 3/5 complete
  Fold 4/5 complete
  Fold 5/5 complete

Top 15 features (by mean SHAP importance across folds):
BMI                                             0.136039
Age                                             0.074622
education_less than high school graduate        0.009923
family_pir                                      0.007478
ever_smoker_yes                                 0.006255
education_high school graduate or equivalent    0.005575
Free_Maltohexaose                               0.004851
Free_Raffinose                                  0.004785
ethnicity_Non-Hispanic_White                    0.004547
Mono_GalA                                       0.004203
Free_Stac

In [2]:
summary_df

,mean_spearman_rho,n_stable_top10
mets,0.732143,5.0
crp,0.802857,5.0
ir,0.586786,5.0
